# 118종 알약 crop 데이터셋 확인

조합 이미지(Kaggle + AI Hub 조합)에서 bbox대로 잘라 낸 crop 데이터를 확인한다.

만드는 순서 (프로젝트 루트에서):
```bash
python -m scripts.classifier_data.validate_labels           # 라벨 검수 → data/classifier/meta/label_issues.csv
python -m scripts.classifier_data.select_images             # 이미지 선정 → data/classifier/meta
python -m scripts.classifier_data.make_crops                # crop 생성 → data/classifier/crops/<약 코드>/
python -m scripts.classifier_data.split_dataset             # train/val 분할 → crops.csv의 split 열
python -m scripts.classifier_data.count_crops               # 클래스별 개수 → crop_counts.csv
python -m scripts.classifier_data.export_classifier_inputs  # 분류기 입력 패키지 → data/classifier/export/
```

train/val 분할 결과는 `reports/data/classifier/split_report.md`에 있다. 5절에서 분류기 파트에 넘기는 입력이 「분류기 데이터 및 함수 입력 명세서」 형식인지 확인한다.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from PIL import Image

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
from scripts.classifier_data.paths import CLASSES, CROPS, META

plt.rcParams["font.family"] = "AppleGothic"   # 한글 표시 (macOS)
plt.rcParams["axes.unicode_minus"] = False
pd.set_option("display.max_rows", 130)

classes = pd.read_csv(CLASSES)
crops = pd.read_csv(CROPS / "crops.csv")
counts = pd.read_csv(CROPS / "crop_counts.csv", index_col="code")
summary = pd.read_csv(META / "class_summary.csv")
print(f"클래스 {len(classes)}종, crop {len(crops)}개, 원본 이미지 {crops.image.nunique()}장")

## 1. 클래스별 crop 개수

In [ ]:
TARGET = 108
print(f"클래스별 최소 {counts.total.min()}, 중앙값 {counts.total.median():.0f}, 최대 {counts.total.max()}")
print(f"{TARGET}개 미만 {(counts.total < TARGET).sum()}종, {TARGET}개 정확히 {(counts.total == TARGET).sum()}종, 초과 {(counts.total > TARGET).sum()}종")

c = counts.sort_values("total")
fig, ax = plt.subplots(figsize=(14, 4))
ax.bar(range(len(c)), c.combo, label="AI Hub 조합", color="#4C78A8")
ax.bar(range(len(c)), c.kaggle, bottom=c.combo, label="Kaggle", color="#F58518")
ax.axhline(TARGET, color="gray", ls="--", lw=1, label=f"목표 {TARGET}")
ax.set_xlabel("클래스 (개수 적은 순)"); ax.set_ylabel("crop 수"); ax.set_xticks([])
ax.legend(); plt.tight_layout(); plt.show()

**해석**: crop 14988개(원본 이미지 4048장), 클래스별 중앙값 108이고 80종이 정확히 108개다. 목표 미만 8종, 초과 30종이다.

### 목표에 못 미치는 클래스 (있는 만큼 사용)

In [ ]:
# combo_available: 검수 후 남은 조합 bbox 수 (선정 후보 전체)
counts[counts.total < TARGET][["class_id", "name", "total", "combo", "kaggle"]].join(summary.set_index("code").combo_available)

**해석**: 목표 미만 8종 중 7종은 Kaggle 0개에 조합 79~106개이고, combo와 combo_available이 같아 검수 후 남은 조합 bbox를 모두 썼다. 기넥신(K-003483)은 조합 후보가 0개라 Kaggle 45개뿐이다. 이번 데이터셋에서는 있는 만큼 쓴다.

### 목표를 크게 넘는 클래스
대부분 다른 약과 같은 조합에 함께 찍혀 있어, 부족한 클래스를 채우는 과정에서 같이 늘어난 경우다. 하이트린(K-003351)은 조합 이미지가 없고 Kaggle에만 있는데, Kaggle은 전부 쓰므로 Kaggle 개수 그대로다.

In [ ]:
counts[counts.total > 150][["class_id", "name", "total", "combo", "kaggle"]].sort_values("total", ascending=False)

## 2. 촬영 각도 분포

In [ ]:
img = crops.drop_duplicates("image").assign(angle=lambda d: d.image.str.split("_").str[5])
pd.crosstab(img.angle, img.source, margins=True)

**해석**: 선정된 원본 이미지 4048장에서 각도 70/75/90°가 1318/1352/1378장으로 고르게 섞였다. 조합(1246/1277/1293)과 Kaggle(72/75/85)도 각각 비슷한 비율이다.

## 3. 제외한 데이터와 crop 모양
- 라벨 검수(`validate_labels.py`, 「알약 데이터 검증 체크리스트」 기준)에서 오류로 판단한 것은 선정 전에 뺐다. 조합은 이미지째, Kaggle은 해당 bbox만 뺐다
- crop은 bbox 사방 5% 여백을 이미지 경계 안에서 자르고, 짧은 쪽을 고정색 (124, 116, 104)으로 채워 정사각형으로 만들었다 (`pilldata/transforms.py`). 여기서 보는 것은 확인용 JPEG crop이고, 분류기 학습 패키지는 padding 전 상태로 저장해 로더가 회전·padding을 한다 (5절)
- 항목별 전체 기록: `reports/data/classifier/validation_report.md`

In [ ]:
issues = pd.read_csv(META / "label_issues.csv")
drop = issues[issues.action.isin(["이미지 제외", "bbox 제외"])]
print(f"이미지 제외: 조합 {drop[(drop.action == '이미지 제외') & (drop.source == 'combo')].image.nunique()}장, "
      f"Kaggle {drop[(drop.action == '이미지 제외') & (drop.source == 'kaggle')].image.nunique()}장 / "
      f"bbox 제외 {len(drop[drop.action == 'bbox 제외'][['image', 'code']].drop_duplicates())}개")
print(f"선정된 이미지 중 제외 대상: {crops.image.isin(drop[drop.action == '이미지 제외'].image).sum()}개")

sizes = pd.Series([Image.open(CROPS / f).size for f in crops.crop])
print(f"정사각형이 아닌 crop: {(sizes.map(lambda s: s[0] != s[1])).sum()}개 / {len(sizes)}개")
drop.groupby(["check", "source"]).image.nunique().unstack(fill_value=0)

**해석**: 검수에서 조합 393장을 이미지째, Kaggle bbox 7개를 뺐고, 선정된 이미지 중 제외 대상은 0개라 제외가 제대로 반영됐다. 사유별로는 bbox 음수·0 크기(조합 122장)가 가장 많고, bbox 크기 ↔ 실제 알약 크기 불일치(조합 112장, Kaggle 1장, 짧은 변 검사와 크기 값이 없는 약의 클래스 중앙값 비교 포함), 동일 bbox 반복(조합 68장, Kaggle 3장), 파일명 ↔ 라벨 구성 불일치(조합 58장)가 그다음이다. bbox 과도한 중첩은 작은 bbox가 다른 bbox 안에 들어간 경우까지 세어 조합 34장이다. 한 이미지가 여러 사유에 걸릴 수 있어 사유별 합은 전체보다 크다. crop 14988개 모두 정사각형이다.

## 4. crop 샘플
목표 미만 클래스와 많은 클래스에서 몇 장씩 본다.

In [ ]:
show = list(counts.sort_values("total").index[:4]) + list(counts.sort_values("total").index[-2:])
fig, axes = plt.subplots(len(show), 6, figsize=(12, 2.2 * len(show)))
for row, code in zip(axes, show):
    files = crops[crops.code == code].sample(6, random_state=0).crop
    for ax, f in zip(row, files):
        ax.imshow(Image.open(CROPS / f)); ax.axis("off")
    row[0].set_title(f"{code} {counts.loc[code, 'name'][:12]}", loc="left", fontsize=9)
plt.tight_layout(); plt.show()

## 5. 분류기 입력 확인 (명세서 형식)
분류기 파트에 넘기는 `build_classifier_inputs`(`classifier_inputs.py`)가 명세서 9~10절 조건을 통과하는지 본다. `export_classifier_inputs.py`로 만든 패키지를 읽는다.

In [ ]:
from types import SimpleNamespace

import torch

from classifier.training_data import validate_classifier_training_data
from pilldata.classifier_dataset import MEAN, STD, build_classifier_inputs
from scripts.classifier_data.paths import EXPORT

d = EXPORT / "training_package_classifier"
cfg = SimpleNamespace(batch_size=16, seed=42, class_map_path=d / "class_map.json", data_summary_path=d / "data_summary.json")
train_loader, val_loader, class_map, data_summary = build_classifier_inputs(config=cfg, device=torch.device("cpu"))
validate_classifier_training_data(
    (train_loader, val_loader, class_map, data_summary),
    expected_num_classes=class_map["num_classes"],
)
print("입력 검증 완료 (명세서 9~10절 조건)")
print(f"클래스 {class_map['num_classes']}종, train {data_summary['num_train_samples']}개 / val {data_summary['num_val_samples']}개, "
      f"그룹 train {data_summary['train_group_count']}개 / val {data_summary['val_group_count']}개, 겹침 {data_summary['group_overlap_count']}개")

b = next(iter(val_loader))
x, y, meta = b["images"], b["targets"], b["meta"]
print("images:", x.dtype, tuple(x.shape), "/ targets:", y.dtype, tuple(y.shape), "/ meta:", type(meta).__name__, len(meta))
print("meta 예:", meta[0])

# val 로더가 모델에 넘기는 이미지 (중앙 padding → 224 resize → 정규화, 무작위 처리 없음)
mean, std = torch.tensor(MEAN).view(3, 1, 1), torch.tensor(STD).view(3, 1, 1)
fig, axes = plt.subplots(2, 8, figsize=(14, 4))
for ax, im, m in zip(axes.flat, x, meta):
    ax.imshow((im * std + mean).clamp(0, 1).permute(1, 2, 0)); ax.axis("off")
    ax.set_title(f"K-{m['category_id']:06d}", fontsize=8)
plt.tight_layout(); plt.show()

**해석**: 명세서 9~10절 조건 검사를 모두 통과했다. 118종, train 12125개 / val 2863개이고 train·val 그룹(1786개 / 429개)은 겹치지 않는다. batch는 `images` float32 (16, 3, 224, 224), `targets` int64 (16,), `meta`는 길이 16의 list이며, 샘플마다 명세서 3.3의 8개 필드가 들어 있다(`item_seq`는 문자열). 그림은 val 이미지로, 회전 없이 중앙 padding·resize·정규화만 적용됐다.